In [1]:
import sys
print("Python:", sys.version.split()[0])

for name in ["pandas", "numpy", "requests", "sklearn", "statsmodels", "matplotlib"]:
    try:
        module = __import__(name)
        print(f"{name}: {module.__version__}")
    except ImportError:
        print(f"{name}: NOT INSTALLED")

Python: 3.12.12
pandas: 3.0.0
numpy: 2.4.2
requests: 2.32.5
sklearn: 1.7.1
statsmodels: 0.14.6
matplotlib: 3.10.8


In [2]:
import requests
import pandas as pd

BASE = "https://api.nhle.com/stats/rest/en"

def get_report(entity, report, season, game_type):
    """Pull every row of one NHL stats report for one season.
    game_type: 2 = regular season, 3 = playoffs."""
    rows, start = [], 0
    while True:
        params = {
            "isAggregate": "false", "isGame": "false",
            "start": start, "limit": 100,
            "sort": '[{"property":"playerId","direction":"ASC"}]',
            "cayenneExp": f"seasonId={season} and gameTypeId={game_type}",
        }
        r = requests.get(f"{BASE}/{entity}/{report}", params=params, timeout=30)
        r.raise_for_status()
        js = r.json()
        rows += js["data"]
        start += 100
        if start >= js["total"]:
            break
    return pd.DataFrame(rows)

test = get_report("skater", "summary", 20232024, 2)
print(test.shape)
test.sort_values("points", ascending=False)[
    ["skaterFullName", "teamAbbrevs", "positionCode", "gamesPlayed", "goals", "assists", "points"]
].head()

(924, 26)


,skaterFullName,teamAbbrevs,positionCode,gamesPlayed,goals,assists,points
168,Nikita Kucherov,TBL,R,81,44,100,144
285,Nathan MacKinnon,COL,C,82,51,89,140
380,Connor McDavid,EDM,C,76,32,100,132
428,Artemi Panarin,NYR,L,82,49,71,120
333,David Pastrnak,BOS,R,82,47,63,110


In [3]:
realtime = get_report("skater", "realtime", 20232024, 2)
toi      = get_report("skater", "timeonice", 20232024, 2)
bios     = get_report("skater", "bios", 20232024, 2)

for name, df in [("realtime", realtime), ("timeonice", toi), ("bios", bios)]:
    print(name, df.shape)
    print(list(df.columns), "\n")

# Traded player check: Jake Guentzel went from Pittsburgh to Carolina in 2023-24
test[test["skaterFullName"] == "Jake Guentzel"][
    ["skaterFullName", "teamAbbrevs", "gamesPlayed", "goals", "assists", "points"]
]

realtime (924, 31)
['blockedShots', 'blockedShotsPer60', 'emptyNetAssists', 'emptyNetGoals', 'emptyNetPoints', 'firstGoals', 'gamesPlayed', 'giveaways', 'giveawaysPer60', 'hits', 'hitsPer60', 'lastName', 'missedShotCrossbar', 'missedShotFailedBankAttempt', 'missedShotGoalpost', 'missedShotOverNet', 'missedShotShort', 'missedShotWideOfNet', 'missedShots', 'otGoals', 'playerId', 'positionCode', 'seasonId', 'shootsCatches', 'shotAttemptsBlocked', 'skaterFullName', 'takeaways', 'takeawaysPer60', 'teamAbbrevs', 'timeOnIcePerGame', 'totalShotAttempts'] 

timeonice (924, 21)
['evTimeOnIce', 'evTimeOnIcePerGame', 'gamesPlayed', 'lastName', 'otTimeOnIce', 'otTimeOnIcePerOtGame', 'playerId', 'positionCode', 'ppTimeOnIce', 'ppTimeOnIcePerGame', 'seasonId', 'shTimeOnIce', 'shTimeOnIcePerGame', 'shifts', 'shiftsPerGame', 'shootsCatches', 'skaterFullName', 'teamAbbrevs', 'timeOnIce', 'timeOnIcePerGame', 'timeOnIcePerShift'] 

bios (924, 23)
['assists', 'birthCity', 'birthCountryCode', 'birthDate', '

,skaterFullName,teamAbbrevs,gamesPlayed,goals,assists,points
260,Jake Guentzel,"PIT,CAR",67,30,47,77


In [4]:
print(list(test.columns), "\n")

season = (test
    .merge(realtime[["playerId", "hits", "blockedShots"]], on="playerId")
    .merge(toi[["playerId", "timeOnIce", "ppTimeOnIce", "shTimeOnIce"]], on="playerId")
    .merge(bios[["playerId", "height", "weight"]], on="playerId"))
print(season.shape)

season[season["skaterFullName"].isin(["Nathan MacKinnon", "Ryan Reaves"])][
    ["skaterFullName", "gamesPlayed", "timeOnIce", "ppTimeOnIce", "shTimeOnIce", "height", "weight"]
]

['assists', 'evGoals', 'evPoints', 'faceoffWinPct', 'gameWinningGoals', 'gamesPlayed', 'goals', 'lastName', 'otGoals', 'penaltyMinutes', 'playerId', 'plusMinus', 'points', 'pointsPerGame', 'positionCode', 'ppGoals', 'ppPoints', 'seasonId', 'shGoals', 'shPoints', 'shootingPct', 'shootsCatches', 'shots', 'skaterFullName', 'teamAbbrevs', 'timeOnIcePerGame'] 

(924, 33)


,skaterFullName,gamesPlayed,timeOnIce,ppTimeOnIce,shTimeOnIce,height,weight
19,Ryan Reaves,49,23862,143,21,74,225
285,Nathan MacKinnon,82,112246,22158,305,72,200


In [5]:
hours = season["timeOnIce"] / 3600   # seconds -> hours, for per-60 rates

season["pos_group"]  = season["positionCode"].map({"C": "F", "L": "F", "R": "F", "D": "D"})
season["pp_share"]   = season["ppTimeOnIce"] / season["timeOnIce"]
season["pk_share"]   = season["shTimeOnIce"] / season["timeOnIce"]
for stat, col in [("goals", "goals"), ("assists", "assists"), ("shots", "shots"),
                  ("hits", "hits"), ("pim", "penaltyMinutes"), ("blocks", "blockedShots")]:
    season[f"{stat}_per60"] = season[col] / hours

features = ["height", "weight", "pp_share", "pk_share", "goals_per60", "assists_per60",
            "shots_per60", "hits_per60", "pim_per60", "blocks_per60"]

print(season["pos_group"].value_counts(dropna=False), "\n")
season[season["skaterFullName"].isin(["Nathan MacKinnon", "Ryan Reaves", "Cale Makar"])][
    ["skaterFullName", "pos_group"] + features
].round(2)

pos_group
F    609
D    315
Name: count, dtype: int64 



,skaterFullName,pos_group,height,weight,pp_share,pk_share,goals_per60,assists_per60,shots_per60,hits_per60,pim_per60,blocks_per60
19,Ryan Reaves,F,74,225,0.01,0.0,0.60,0.30,3.92,24.89,7.39,2.56
285,Nathan MacKinnon,F,72,200,0.20,0.0,1.64,2.85,12.99,1.76,1.35,2.21
572,Cale Makar,D,72,187,0.16,0.1,0.66,2.17,7.27,1.16,0.50,4.66


In [6]:
import time

def build_season(season_id):
    summ = get_report("skater", "summary",   season_id, 2)
    rt   = get_report("skater", "realtime",  season_id, 2)
    ti   = get_report("skater", "timeonice", season_id, 2)
    bio  = get_report("skater", "bios",      season_id, 2)
    return (summ
        .merge(rt[["playerId", "hits", "blockedShots"]], on="playerId", how="left")
        .merge(ti[["playerId", "timeOnIce", "ppTimeOnIce", "shTimeOnIce"]], on="playerId", how="left")
        .merge(bio[["playerId", "height", "weight"]], on="playerId", how="left"))

seasons = [int(f"{y}{y+1}") for y in range(2005, 2026)]
all_seasons = []
for sid in seasons:
    df = build_season(sid)
    missing = df[["hits", "timeOnIce", "height", "weight"]].isna().any(axis=1).sum()
    print(sid, "| players:", len(df), "| rows with a missing value:", missing)
    all_seasons.append(df)
    time.sleep(1)   # brief pause between seasons to be polite to the API

reg = pd.concat(all_seasons, ignore_index=True)
reg.to_csv("regular_season_raw.csv", index=False)
print("\nTotal:", reg.shape)

20052006 | players: 870 | rows with a missing value: 0
20062007 | players: 858 | rows with a missing value: 0
20072008 | players: 852 | rows with a missing value: 0
20082009 | players: 885 | rows with a missing value: 0
20092010 | players: 879 | rows with a missing value: 0
20102011 | players: 891 | rows with a missing value: 0
20112012 | players: 894 | rows with a missing value: 0
20122013 | players: 839 | rows with a missing value: 0
20132014 | players: 886 | rows with a missing value: 0
20142015 | players: 882 | rows with a missing value: 0
20152016 | players: 898 | rows with a missing value: 0
20162017 | players: 888 | rows with a missing value: 0
20172018 | players: 890 | rows with a missing value: 0
20182019 | players: 906 | rows with a missing value: 0
20192020 | players: 883 | rows with a missing value: 0
20202021 | players: 913 | rows with a missing value: 0
20212022 | players: 1004 | rows with a missing value: 0
20222023 | players: 951 | rows with a missing value: 0
20232024 

In [7]:
reg = pd.read_csv("regular_season_raw.csv")

def add_features(df):
    df = df.copy()
    hours = df["timeOnIce"] / 3600
    df["pos_group"] = df["positionCode"].map({"C": "F", "L": "F", "R": "F", "D": "D"})
    df["pp_share"]  = df["ppTimeOnIce"] / df["timeOnIce"]
    df["pk_share"]  = df["shTimeOnIce"] / df["timeOnIce"]
    for stat, col in [("goals", "goals"), ("assists", "assists"), ("shots", "shots"),
                      ("hits", "hits"), ("pim", "penaltyMinutes"), ("blocks", "blockedShots")]:
        df[f"{stat}_per60"] = df[col] / hours
    return df

reg = add_features(reg)
reg["meets_cutoff"] = reg["gamesPlayed"] >= 20

print("Players with zero ice time:", (reg["timeOnIce"] == 0).sum())
print("Unrecognized positions:", reg["pos_group"].isna().sum(), "\n")
print(reg.groupby("pos_group")["meets_cutoff"].agg(total="count", meet_cutoff="sum"), "\n")

reg[reg["meets_cutoff"]].groupby("pos_group")[features].agg(["median", "max"]).T.round(2)

Players with zero ice time: 0
Unrecognized positions: 0 

           total  meet_cutoff
pos_group                    
D           6472         4902
F          12381         9325 



pos_group                  D       F
height        median   74.00   73.00
              max      81.00   81.00
weight        median  207.00  200.00
              max     265.00  268.00
pp_share      median    0.04    0.10
              max       0.48    0.44
pk_share      median    0.09    0.04
              max       0.30    0.44
goals_per60   median    0.17    0.72
              max       1.14    2.44
assists_per60 median    0.66    0.97
              max       2.22    4.28
shots_per60   median    3.93    6.89
              max      12.67   17.43
hits_per60    median    3.68    4.12
              max      22.27   33.45
pim_per60     median    1.60    1.70
              max      37.50   82.72
blocks_per60  median    4.29    1.72
              max      10.68    8.49

In [8]:
po_list = []
for sid in seasons:
    df = get_report("skater", "timeonice", sid, 3)
    df = df[["playerId", "skaterFullName", "positionCode", "teamAbbrevs",
             "seasonId", "gamesPlayed", "timeOnIce"]]
    teams = df["teamAbbrevs"].nunique()
    multi = df["teamAbbrevs"].str.contains(",").sum()
    print(sid, "| players:", len(df), "| teams:", teams, "| multi-team rows:", multi)
    po_list.append(df)
    time.sleep(1)

po = pd.concat(po_list, ignore_index=True)
po.to_csv("playoff_toi_raw.csv", index=False)

check = po.merge(reg[["playerId", "seasonId"]], on=["playerId", "seasonId"], how="left", indicator=True)
print("\nTotal:", po.shape)
print("Playoff rows with no regular-season row:", (check["_merge"] == "left_only").sum())

20052006 | players: 333 | teams: 16 | multi-team rows: 0
20062007 | players: 331 | teams: 16 | multi-team rows: 0
20072008 | players: 340 | teams: 16 | multi-team rows: 0
20082009 | players: 333 | teams: 16 | multi-team rows: 0
20092010 | players: 342 | teams: 16 | multi-team rows: 0
20102011 | players: 339 | teams: 16 | multi-team rows: 0
20112012 | players: 347 | teams: 16 | multi-team rows: 0
20122013 | players: 349 | teams: 16 | multi-team rows: 0
20132014 | players: 344 | teams: 16 | multi-team rows: 0
20142015 | players: 339 | teams: 16 | multi-team rows: 0
20152016 | players: 340 | teams: 16 | multi-team rows: 0
20162017 | players: 343 | teams: 16 | multi-team rows: 0
20172018 | players: 333 | teams: 16 | multi-team rows: 0
20182019 | players: 333 | teams: 16 | multi-team rows: 0
20192020 | players: 516 | teams: 24 | multi-team rows: 0
20202021 | players: 337 | teams: 16 | multi-team rows: 0
20212022 | players: 335 | teams: 16 | multi-team rows: 0
20222023 | players: 334 | teams

In [9]:
missing = check[check["_merge"] == "left_only"].copy()
missing["toi_min"] = (missing["timeOnIce"] / 60).round(1)

team_toi = po.groupby(["seasonId", "teamAbbrevs"])["timeOnIce"].sum().rename("team_toi")
missing = missing.merge(team_toi, on=["seasonId", "teamAbbrevs"])
missing["share_of_team"] = (missing["timeOnIce"] / missing["team_toi"]).round(3)

print("Share of ALL playoff ice time:", round(missing["timeOnIce"].sum() / po["timeOnIce"].sum(), 4))
print("Largest share on any one team:", missing["share_of_team"].max(), "\n")

missing.sort_values("share_of_team", ascending=False)[
    ["skaterFullName", "seasonId", "teamAbbrevs", "positionCode", "gamesPlayed", "toi_min", "share_of_team"]
].head(15)

Share of ALL playoff ice time: 0.0024
Largest share on any one team: 0.08 



,skaterFullName,seasonId,teamAbbrevs,positionCode,gamesPlayed,toi_min,share_of_team
26,Charlie McAvoy,20162017,BOS,D,6,157.2,0.080
38,Nikita Kucherov,20202021,TBL,R,23,420.8,0.062
47,Evander Kane,20242025,EDM,L,21,346.0,0.050
30,Cale Makar,20182019,COL,D,10,173.7,0.048
48,Gabriel Landeskog,20242025,COL,L,5,86.1,0.039
13,Chris Kreider,20112012,NYR,L,18,236.7,0.038
37,Nick Robertson,20192020,TOR,L,4,48.3,0.030
3,Jannik Hansen,20062007,VAN,R,10,126.9,0.030
45,Tye Kartye,20222023,SEA,L,10,123.4,0.029
50,Zeev Buium,20242025,MIN,D,4,54.4,0.029


In [10]:
reg_keys = reg[["playerId", "seasonId"]]

def nearest_season(player_id, season_id):
    s = reg_keys.loc[reg_keys["playerId"] == player_id, "seasonId"]
    prior = s[s < season_id]
    if len(prior):
        return prior.max()      # most recent earlier season
    later = s[s > season_id]
    if len(later):
        return later.min()      # otherwise, the next season
    return None                 # no regular-season data at all

po["feat_season"] = po["seasonId"].astype(float)
no_reg = check["_merge"].eq("left_only").to_numpy()
po.loc[no_reg, "feat_season"] = [nearest_season(p, s) for p, s in
                                 zip(po.loc[no_reg, "playerId"], po.loc[no_reg, "seasonId"])]

print("Rows using a borrowed season:", no_reg.sum())
print("Rows that couldn't be matched:", po["feat_season"].isna().sum(), "\n")

po.to_csv("playoff_toi.csv", index=False)
po.loc[no_reg, ["skaterFullName", "seasonId", "teamAbbrevs", "feat_season"]].head(10)

Rows using a borrowed season: 51
Rows that couldn't be matched: 0 



,skaterFullName,seasonId,teamAbbrevs,feat_season
218,Toby Petersen,20052006,EDM,20062007.0
231,Doug Janik,20052006,BUF,20062007.0
640,Drew Miller,20062007,ANA,20072008.0
656,Jannik Hansen,20062007,VAN,20072008.0
663,Ryan Carter,20062007,ANA,20072008.0
872,Erik Reitz,20072008,MIN,20062007.0
959,Lauri Korpikoski,20072008,NYR,20082009.0
1042,Anders Eriksson,20082009,CGY,20072008.0
1079,Josh Green,20082009,ANA,20062007.0
1285,Torrey Mitchell,20082009,SJS,20072008.0


In [11]:
def get_team_report(report, season, game_type):
    params = {
        "isAggregate": "false", "isGame": "false", "start": 0, "limit": 50,
        "sort": '[{"property":"teamId","direction":"ASC"}]',
        "cayenneExp": f"seasonId={season} and gameTypeId={game_type}",
    }
    r = requests.get(f"{BASE}/team/{report}", params=params, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

t_reg = get_team_report("summary", 20232024, 2)
t_po  = get_team_report("summary", 20232024, 3)
print("regular season:", t_reg.shape, "| playoffs:", t_po.shape)
print(list(t_reg.columns), "\n")

t_po[["teamFullName", "gamesPlayed", "wins", "losses"]].sort_values("wins", ascending=False)

regular season: (32, 25) | playoffs: (16, 25)
['faceoffWinPct', 'gamesPlayed', 'goalsAgainst', 'goalsAgainstPerGame', 'goalsFor', 'goalsForPerGame', 'losses', 'otLosses', 'penaltyKillNetPct', 'penaltyKillPct', 'pointPct', 'points', 'powerPlayNetPct', 'powerPlayPct', 'regulationAndOtWins', 'seasonId', 'shotsAgainstPerGame', 'shotsForPerGame', 'teamFullName', 'teamId', 'teamShutouts', 'ties', 'wins', 'winsInRegulation', 'winsInShootout'] 



,teamFullName,gamesPlayed,wins,losses
5,Florida Panthers,24,16,8
10,Edmonton Oilers,25,15,10
12,Dallas Stars,19,10,9
1,New York Rangers,16,10,6
11,Vancouver Canucks,13,7,6
9,Colorado Avalanche,11,6,5
4,Carolina Hurricanes,11,6,5
2,Boston Bruins,13,6,7
15,Vegas Golden Knights,7,3,4
3,Toronto Maple Leafs,7,3,4


In [12]:
team_map = pd.DataFrame(requests.get(f"{BASE}/team", timeout=30).json()["data"])
team_map = team_map[["id", "triCode"]].rename(columns={"id": "teamId"})

team_rows = []
for sid in seasons:
    tr = get_team_report("summary", sid, 2)
    tp = get_team_report("summary", sid, 3)
    tr = tr[["teamId", "teamFullName", "seasonId", "gamesPlayed",
             "pointPct", "goalsAgainst", "shotsAgainstPerGame"]].copy()
    tr["save_pct"] = 1 - tr["goalsAgainst"] / (tr["shotsAgainstPerGame"] * tr["gamesPlayed"])
    tp = tp[["teamId", "wins"]].rename(columns={"wins": "playoff_wins"})
    t = tp.merge(tr, on="teamId", how="left").merge(team_map, on="teamId", how="left")
    t["rounds_won"] = t["playoff_wins"] // 4
    team_rows.append(t)

    api_codes = set(t["triCode"])
    player_codes = set(po.loc[po["seasonId"] == sid, "teamAbbrevs"])
    if api_codes != player_codes:
        print(sid, "MISMATCH | team data only:", api_codes - player_codes,
              "| player data only:", player_codes - api_codes)
    time.sleep(1)

team = pd.concat(team_rows, ignore_index=True)
team.to_csv("team_raw.csv", index=False)
print("\nTotal:", team.shape)
print("Missing values:", int(team.isna().sum().sum()))
print(team["save_pct"].describe().round(3))


Total: (344, 11)
Missing values: 0
count    344.000
mean       0.909
std        0.010
min        0.876
25%        0.903
50%        0.910
75%        0.915
max        0.933
Name: save_pct, dtype: float64


In [13]:
rounds_2020 = {"TBL": 4, "DAL": 3, "VGK": 2, "NYI": 2,
               "COL": 1, "VAN": 1, "PHI": 1, "BOS": 1,
               "CHI": 0, "ARI": 0, "CGY": 0, "STL": 0,
               "MTL": 0, "CBJ": 0, "WSH": 0, "CAR": 0}

is_2020 = team["seasonId"] == 20192020
dropped = sorted(set(team.loc[is_2020, "triCode"]) - set(rounds_2020))
print("2019-20 qualifying-round losers dropped:", dropped)

team = team[~is_2020 | team["triCode"].isin(rounds_2020)].copy()
is_2020 = team["seasonId"] == 20192020
team.loc[is_2020, "rounds_won"] = team.loc[is_2020, "triCode"].map(rounds_2020)

check_t = team.groupby("seasonId")["rounds_won"].agg(teams="count", total="sum", champ="max")
bad = check_t[(check_t["teams"] != 16) | (check_t["total"] != 15) | (check_t["champ"] != 4)]
print("Total rows:", len(team))
print("Seasons failing the check:", len(bad))
print(bad)

team.to_csv("team.csv", index=False)

2019-20 qualifying-round losers dropped: ['EDM', 'FLA', 'MIN', 'NSH', 'NYR', 'PIT', 'TOR', 'WPG']
Total rows: 336
Seasons failing the check: 0
Empty DataFrame
Columns: [teams, total, champ]
Index: []
